# Modulo 1: Architettura di un Agente Autonomo per l'Analisi delle Frodi E-commerce

## Obiettivi del Notebook
In questo notebook costruirai da zero un **Autonomous Fraud Detection Agent** in grado di investigare transazioni e-commerce sospette e mitigare rischi finanziari.

Imparerai a:
1. **Comprendere l'architettura agentica**: Differenza tra LLM semplice, Workflow rigido e Agente autonomo ReAct (*Reason $\rightarrow$ Act $\rightarrow$ Observe*).
2. **Definire Tool Contracts sicuri**: Scrivere funzioni Python per l'analisi del rischio con gestione strutturata degli errori.
3. **Implementare un Agent Loop**: Gestire la memoria di lavoro, il budget di turni (max steps) e la condizione di terminazione.
4. **Applicare il pattern Human-in-the-Loop (HitL)**: Separare le operazioni di sola lettura da quelle con effetto collaterale (*side-effect* come il blocco di un ordine o account).
5. **Analizzare la Trace e la Grounding**: Verificare che le decisioni dell'agente siano supportate da evidenze concrete.
6. **Integrare le API di OpenAI (Function Calling)**: Collegare l'agente a un modello linguistico reale.

### Setup & Inizializzazione

In [13]:
port os
import json
import time
from datetime import datetime
from typing import Dict, Any, List, Optional, Callable
from dataclasses import dataclass, field
import nest_asyncio

# Abilita l'esecuzione asincrona all'interno dei notebook Jupyter
nest_asyncio.apply()

print("✅ Ambiente inizializzato con successo.")

✅ Ambiente inizializzato con successo.


## Sezione 1: Architettura dell'Agente per la Prevenzione Frodi

L'agente opera in un ambiente e-commerce e deve gestire incidenti di sicurezza finanziaria.

### Mappatura dei Componenti:
- **Environment (Ambiente):** Database ordini, registro utenti e log di geolocalizzazione IP.
- **Tools (Strumenti):**
  - `check_customer_history`: Legge lo storico acquisti e chargeback dell'utente (Read-only).
  - `verify_geo_ip`: Analizza il rischio legato all'indirizzo IP dell'ordine (Read-only).
  - `flag_and_freeze_order`: Congela l'ordine e sospende la spedizione (Side-effect $\rightarrow$ Richiede approvazione umana).
- **State (Stato / Memoria):** Storico dei passaggi del ciclo ReAct e osservazioni accumulate.
- **Planner (Ciclo ReAct):** Sequenza iterativa di decisione ed esecuzione degli strumenti.

### Simulazione Database & Registri E-commerce

In [14]:
# Simulated Databases for E-commerce Operations
ORDERS_DB: Dict[str, Dict[str, Any]] = {
    "ORD-9901": {
        "order_id": "ORD-9901",
        "user_id": "USR-4412",
        "amount": 2499.00,
        "currency": "EUR",
        "ip_address": "185.220.101.5",
        "shipping_country": "IT",
        "status": "PENDING_CHECK",
        "created_at": "2026-09-28T14:20:00Z"
    },
    "ORD-9902": {
        "order_id": "ORD-9902",
        "user_id": "USR-1002",
        "amount": 45.50,
        "currency": "EUR",
        "ip_address": "79.12.44.10",
        "shipping_country": "IT",
        "status": "APPROVED",
        "created_at": "2026-09-28T14:22:00Z"
    }
}

USERS_DB: Dict[str, Dict[str, Any]] = {
    "USR-4412": {
        "user_id": "USR-4412",
        "account_created_minutes_ago": 12,
        "total_past_orders": 0,
        "past_chargebacks": 2,
        "payment_method_country": "US"
    },
    "USR-1002": {
        "user_id": "USR-1002",
        "account_created_minutes_ago": 120000,
        "total_past_orders": 34,
        "past_chargebacks": 0,
        "payment_method_country": "IT"
    }
}

GEO_IP_DB: Dict[str, Dict[str, Any]] = {
    "185.220.101.5": {
        "ip": "185.220.101.5",
        "country": "PA", # Panama
        "is_vpn_or_tor": True,
        "risk_score": 92 # High Risk
    },
    "79.12.44.10": {
        "ip": "79.12.44.10",
        "country": "IT",
        "is_vpn_or_tor": False,
        "risk_score": 5 # Low Risk
    }
}

print("✅ Database simulati caricati.")

✅ Database simulati caricati.


## Sezione 2: Definizione dei Tool Contracts

I tool devono seguire regole rigorose:
1. Input e output tipizzati.
2. Risposte JSON-serializzabili contenenti `status`, `data` ed eventualmente `error`.
3. Gestione pulita delle eccezioni (restituire errori come dati anziché far crollare l'agente).

### Implementazione dei Tool Python


In [15]:
def check_customer_history(user_id: str) -> Dict[str, Any]:
    """
    [READ-ONLY] Recupera lo storico dell'utente dal database.
    """
    try:
        if user_id not in USERS_DB:
            return {
                "status": "error",
                "error": f"Utente {user_id} non trovato nel database.",
                "data": None
            }
        
        user_info = USERS_DB[user_id]
        return {
            "status": "success",
            "data": user_info,
            "error": None
        }
    except Exception as e:
        return {
            "status": "error",
            "error": f"Errore imprevisto: {str(e)}",
            "data": None
        }


def verify_geo_ip(ip_address: str) -> Dict[str, Any]:
    """
    [READ-ONLY] Analizza la geolocalizzazione dell'IP e il punteggio di rischio VPN/proxy.
    """
    try:
        if ip_address not in GEO_IP_DB:
            return {
                "status": "success",
                "data": {
                    "ip": ip_address,
                    "country": "UNKNOWN",
                    "is_vpn_or_tor": False,
                    "risk_score": 50
                },
                "error": None
            }
        
        geo_info = GEO_IP_DB[ip_address]
        return {
            "status": "success",
            "data": geo_info,
            "error": None
        }
    except Exception as e:
        return {
            "status": "error",
            "error": f"Errore durante l'analisi IP: {str(e)}",
            "data": None
        }


def flag_and_freeze_order(order_id: str, reason: str) -> Dict[str, Any]:
    """
    [SIDE-EFFECT] Congela l'ordine sospetto e blocca la spedizione della merce.
    Trattandosi di un'azione ad alto impatto commerciale, richiede approvazione umana (HitL).
    """
    try:
        if order_id not in ORDERS_DB:
            return {
                "status": "error",
                "error": f"Ordine {order_id} non trovato.",
                "data": None
            }
        
        # Aggiorna lo stato nel DB
        ORDERS_DB[order_id]["status"] = "FROZEN_FRAUD_RISK"
        ORDERS_DB[order_id]["freeze_reason"] = reason
        
        return {
            "status": "success",
            "data": {
                "order_id": order_id,
                "new_status": "FROZEN_FRAUD_RISK",
                "reason": reason,
                "timestamp": datetime.utcnow().isoformat()
            },
            "error": None
        }
    except Exception as e:
        return {
            "status": "error",
            "error": f"Impossibile congelare l'ordine: {str(e)}",
            "data": None
        }

print("✅ Tool registrati correttamente.")

✅ Tool registrati correttamente.


## Sezione 3: Definizione dello Stato e del Ciclo ReAct

L'agente mantiene una memoria di lavoro denominata `AgentState`.
Il ciclo ReAct segue questa sequenza:
1. **Reason**: Valuta le osservazioni correnti.
2. **Act**: Sceglie lo strumento ed esegue l'azione.
3. **Observe**: Riceve il risultato del tool e aggiorna lo stato.

### Struttura dello Stato dell'Agente

In [16]:
@dataclass
class ActionStep:
    thought: str
    tool_name: str
    tool_args: Dict[str, Any]
    observation: Optional[Dict[str, Any]] = None

@dataclass
class AgentState:
    order_id: str
    steps: List[ActionStep] = field(default_factory=list)
    is_complete: bool = False
    final_decision: Optional[str] = None
    turn_count: int = 0
    max_turns: int = 5

## Sezione 4: Implementazione dell'Agent Loop con Human-in-the-Loop (HitL)

Per prevenire blocchi errati di clienti legittimi, separiamo i tool in due categorie:
- **Read-Only Tools**: Eseguiti direttamente.
- **Side-Effect Tools**: Intercettati dall'agente, richiedono conferma esplicita dall'operatore prima dell'esecuzione.

### Ciclo di Esecuzione e Gestione Sicurezza

In [7]:
# Mappatura dei tool e della loro pericolosità
TOOL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "check_customer_history": {
        "func": check_customer_history,
        "is_side_effect": False
    },
    "verify_geo_ip": {
        "func": verify_geo_ip,
        "is_side_effect": False
    },
    "flag_and_freeze_order": {
        "func": flag_and_freeze_order,
        "is_side_effect": True # Richiede HitL
    }
}

def execute_agent_step(state: AgentState, planned_step: ActionStep, human_approved: bool = False) -> AgentState:
    """
    Esegue un singolo passo del ciclo agentico gestendo i controlli di sicurezza.
    """
    state.turn_count += 1
    
    # Controlla il budget di turni
    if state.turn_count > state.max_turns:
        state.is_complete = True
        state.final_decision = "ABORTED: Raggiunto il limite massimo di turni (Max turns exceeded)."
        print("⚠️ Budget di turni esaurito!")
        return state

    tool_name = planned_step.tool_name
    
    if tool_name not in TOOL_REGISTRY:
        planned_step.observation = {"status": "error", "error": f"Tool '{tool_name}' sconosciuto."}
        state.steps.append(planned_step)
        return state

    tool_meta = TOOL_REGISTRY[tool_name]
    
    # Controllo Human-in-the-Loop per azioni con effetti collaterali
    if tool_meta["is_side_effect"] and not human_approved:
        print(f"\n🛑 [HitL SECURITY INTERCEPT] L'agente richiede di eseguire un'azione critica: '{tool_name}'")
        print(f"   Parametri: {json.dumps(planned_step.tool_args)}")
        print(f"   Motivazione: {planned_step.thought}")
        
        # In uno scenario d'uso reale, qui si attende l'input dell'utente
        planned_step.observation = {
            "status": "pending_human_approval",
            "message": "Azione bloccata in attesa di autorizzazione dell'operatore di sicurezza."
        }
        state.steps.append(planned_step)
        return state

    # Esecuzione del tool
    print(f"\n⚙️ [EXECUTE TOOL] Esecuzione di '{tool_name}'...")
    result = tool_meta["func"](**planned_step.tool_args)
    planned_step.observation = result
    state.steps.append(planned_step)
    
    return state

print("✅ Agent Loop e modulo HitL configurati.")

✅ Agent Loop e modulo HitL configurati.


## Sezione 5: Esecuzione Simulata dell'Agente di Prevenzione Frodi

Simuliamo l'investigazione sull'ordine **ORD-9901**. 
Vedremo l'agente raccogliere prove (storico utente + geolocalizzazione IP) e proporre il congelamento dell'ordine.

### Simulazione Passo-Passo

In [8]:
# Inizializzazione dello stato per l'ordine ORD-9901
target_order = "ORD-9901"
order_data = ORDERS_DB[target_order]

state = AgentState(order_id=target_order)

print(f"=== INIZIO INVESTIGAZIONE FRODE PER ORDINE: {target_order} ===")
print(f"Dati iniziali ordine: Valore={order_data['amount']} {order_data['currency']}, IP={order_data['ip_address']}, UserID={order_data['user_id']}")

# PASSO 1: L'agente decide di controllare lo storico dell'utente
step1 = ActionStep(
    thought="L'ordine supera i 2000€. Devo verificare lo storico acquisti dell'utente e la presenza di chargeback passati.",
    tool_name="check_customer_history",
    tool_args={"user_id": order_data["user_id"]}
)
state = execute_agent_step(state, step1)
print(f"Osservazione Step 1: {json.dumps(step1.observation, indent=2)}")

# PASSO 2: L'agente analizza l'IP dell'ordine
step2 = ActionStep(
    thought="L'account è stato creato da soli 12 minuti e ha 2 chargeback. Devo verificare la provenienza dell'IP e la presenza di VPN/Tor.",
    tool_name="verify_geo_ip",
    tool_args={"ip_address": order_data["ip_address"]}
)
state = execute_agent_step(state, step2)
print(f"Osservazione Step 2: {json.dumps(step2.observation, indent=2)}")

# PASSO 3: Tentativo di blocco dell'ordine (Richiede HitL)
step3 = ActionStep(
    thought="L'IP proviene da un nodo TOR/VPN a Panama con Rischio 92/100, mentre l'ordine è verso l'Italia e l'account ha precedenti di frode. Procedo con il congelamento preventivo dell'ordine.",
    tool_name="flag_and_freeze_order",
    tool_args={
        "order_id": target_order,
        "reason": "Sospetto utilizzo di credenziali rubate (Account nuovo, IP Tor/VPN, Discrepanza geografica)."
    }
)

# Primo tentativo senza approvazione umana
state = execute_agent_step(state, step3, human_approved=False)

# Simulazione dell'operatore umano che approva l'operazione
print("\n👤 [HUMAN OPERATOR]: L'analista della sicurezza controlla i dati e APPROVA l'operazione.")
state = execute_agent_step(state, step3, human_approved=True)

state.is_complete = True
state.final_decision = f"Ordine {target_order} congelato con successo a causa di un elevato rischio di frode."

print(f"\n=== ESITO INVESTIGAZIONE ===")
print(f"Stato Finale: {state.final_decision}")

=== INIZIO INVESTIGAZIONE FRODE PER ORDINE: ORD-9901 ===
Dati iniziali ordine: Valore=2499.0 EUR, IP=185.220.101.5, UserID=USR-4412

⚙️ [EXECUTE TOOL] Esecuzione di 'check_customer_history'...
Osservazione Step 1: {
  "status": "success",
  "data": {
    "user_id": "USR-4412",
    "account_created_minutes_ago": 12,
    "total_past_orders": 0,
    "past_chargebacks": 2,
    "payment_method_country": "US"
  },
  "error": null
}

⚙️ [EXECUTE TOOL] Esecuzione di 'verify_geo_ip'...
Osservazione Step 2: {
  "status": "success",
  "data": {
    "ip": "185.220.101.5",
    "country": "PA",
    "is_vpn_or_tor": true,
    "risk_score": 92
  },
  "error": null
}

🛑 [HitL SECURITY INTERCEPT] L'agente richiede di eseguire un'azione critica: 'flag_and_freeze_order'
   Parametri: {"order_id": "ORD-9901", "reason": "Sospetto utilizzo di credenziali rubate (Account nuovo, IP Tor/VPN, Discrepanza geografica)."}
   Motivazione: L'IP proviene da un nodo TOR/VPN a Panama con Rischio 92/100, mentre l'ordine 

C:\Users\Allysa\AppData\Local\Temp\ipykernel_26008\1293588318.py:81: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()


## Sezione 5.1: Testing su Scenari Multipli

Per verificare la robustezza dell'agente, testiamo il suo comportamento su 3 scenari reali ad alto impatto di business:
1. **ORD-9903 (Cliente VIP)**: Ordine da 3.800€ con utente storico e IP sicuro.
2. **ORD-9904 (Account Takeover)**: Account affidabile con 18 ordini passati, usato improvvisamente tramite Proxy/VPN dal Vietnam.
3. **ORD-9905 (Falso Positivo / Viaggio)**: Utente regolare con IP francese e VPN (rischio medio).

In [9]:
# 1. Aggiornamento dei Database simulati con i nuovi casi di test
ORDERS_DB.update({
    "ORD-9903": {
        "order_id": "ORD-9903",
        "user_id": "USR-VIP-01",
        "amount": 3800.00,
        "currency": "EUR",
        "ip_address": "82.55.10.12",
        "shipping_country": "IT",
        "status": "PENDING_CHECK",
        "created_at": "2026-09-28T15:00:00Z"
    },
    "ORD-9904": {
        "order_id": "ORD-9904",
        "user_id": "USR-OLD-88",
        "amount": 1150.00,
        "currency": "EUR",
        "ip_address": "103.250.2.1",
        "shipping_country": "VN",
        "status": "PENDING_CHECK",
        "created_at": "2026-09-28T15:05:00Z"
    },
    "ORD-9905": {
        "order_id": "ORD-9905",
        "user_id": "USR-REG-22",
        "amount": 180.00,
        "currency": "EUR",
        "ip_address": "194.28.11.5",
        "shipping_country": "IT",
        "status": "PENDING_CHECK",
        "created_at": "2026-09-28T15:10:00Z"
    }
})

USERS_DB.update({
    "USR-VIP-01": {
        "user_id": "USR-VIP-01",
        "account_created_minutes_ago": 525600,
        "total_past_orders": 85,
        "past_chargebacks": 0,
        "payment_method_country": "IT"
    },
    "USR-OLD-88": {
        "user_id": "USR-OLD-88",
        "account_created_minutes_ago": 262800,
        "total_past_orders": 18,
        "past_chargebacks": 0,
        "payment_method_country": "IT"
    },
    "USR-REG-22": {
        "user_id": "USR-REG-22",
        "account_created_minutes_ago": 43200,
        "total_past_orders": 4,
        "past_chargebacks": 0,
        "payment_method_country": "IT"
    }
})

GEO_IP_DB.update({
    "82.55.10.12": {
        "ip": "82.55.10.12",
        "country": "IT",
        "is_vpn_or_tor": False,
        "risk_score": 2
    },
    "103.250.2.1": {
        "ip": "103.250.2.1",
        "country": "VN",
        "is_vpn_or_tor": True,
        "risk_score": 88
    },
    "194.28.11.5": {
        "ip": "194.28.11.5",
        "country": "FR",
        "is_vpn_or_tor": True,
        "risk_score": 45
    }
})

# 2. Funzione Runner per eseguire la simulazione sui vari ordini
def run_scenario_test(order_id: str):
    order = ORDERS_DB[order_id]
    print(f"\n-------------------------------------------------------")
    print(f"🧪 ESECUZIONE TEST SU ORDINE: {order_id}")
    print(f"   Importo: {order['amount']} {order['currency']} | User: {order['user_id']} | IP: {order['ip_address']}")
    print(f"-------------------------------------------------------")
    
    test_state = AgentState(order_id=order_id)
    
    # STEP 1: Analisi Utente
    step_user = ActionStep(
        thought=f"Verifico lo storico dell'utente {order['user_id']}.",
        tool_name="check_customer_history",
        tool_args={"user_id": order["user_id"]}
    )
    test_state = execute_agent_step(test_state, step_user)
    user_data = step_user.observation["data"]
    
    # STEP 2: Analisi IP
    step_ip = ActionStep(
        thought=f"Analizzo la provenienza dell'IP {order['ip_address']}.",
        tool_name="verify_geo_ip",
        tool_args={"ip_address": order["ip_address"]}
    )
    test_state = execute_agent_step(test_state, step_ip)
    ip_data = step_ip.observation["data"]
    
    # STEP 3: Valutazione Rischio dell'Agente
    is_high_risk_ip = ip_data["risk_score"] > 70
    is_new_or_bad_user = user_data["past_chargebacks"] > 0 or user_data["account_created_minutes_ago"] < 60
    country_mismatch = user_data["payment_method_country"] != order["shipping_country"]
    
    if (is_high_risk_ip and country_mismatch) or (is_high_risk_ip and is_new_or_bad_user):
        print("🚨 RISULTATO: Rischio Frode Elevato / Account Takeover!")
        step_freeze = ActionStep(
            thought="Indicatori critici di frode. Procedo con il congelamento.",
            tool_name="flag_and_freeze_order",
            tool_args={"order_id": order_id, "reason": "Sospetto Account Takeover / IP ad alto rischio."}
        )
        test_state = execute_agent_step(test_state, step_freeze, human_approved=False)
    elif ip_data["risk_score"] > 40:
        print("⚠️ RISULTATO: Rischio Medio (Falso Positivo / Utente in viaggio). Nessun blocco, richiesta 2FA.")
        test_state.final_decision = "ORDINE IN ATTESA DI VERIFICA 2FA"
    else:
        print("✅ RISULTATO: Ordine Legittimo.")
        ORDERS_DB[order_id]["status"] = "APPROVED"
        test_state.final_decision = "ORDINE APPROVATO"

# 3. Invocazione della batteria di test
run_scenario_test("ORD-9903")
run_scenario_test("ORD-9904")
run_scenario_test("ORD-9905")


-------------------------------------------------------
🧪 ESECUZIONE TEST SU ORDINE: ORD-9903
   Importo: 3800.0 EUR | User: USR-VIP-01 | IP: 82.55.10.12
-------------------------------------------------------

⚙️ [EXECUTE TOOL] Esecuzione di 'check_customer_history'...

⚙️ [EXECUTE TOOL] Esecuzione di 'verify_geo_ip'...
✅ RISULTATO: Ordine Legittimo.

-------------------------------------------------------
🧪 ESECUZIONE TEST SU ORDINE: ORD-9904
   Importo: 1150.0 EUR | User: USR-OLD-88 | IP: 103.250.2.1
-------------------------------------------------------

⚙️ [EXECUTE TOOL] Esecuzione di 'check_customer_history'...

⚙️ [EXECUTE TOOL] Esecuzione di 'verify_geo_ip'...
🚨 RISULTATO: Rischio Frode Elevato / Account Takeover!

🛑 [HitL SECURITY INTERCEPT] L'agente richiede di eseguire un'azione critica: 'flag_and_freeze_order'
   Parametri: {"order_id": "ORD-9904", "reason": "Sospetto Account Takeover / IP ad alto rischio."}
   Motivazione: Indicatori critici di frode. Procedo con il cong

## Sezione 6: Analisi della Trace e Audit Trail

In ambito finanziario e di sicurezza, la spiegabilità (*Explainability*) è un requisito fondamentale.
Analizziamo la traccia delle decisioni prese dall'agente.

### Audit Trail Format

In [10]:
print("=================== AUDIT TRACE LOG ===================")
for idx, s in enumerate(state.steps, 1):
    print(f"\n--- TURN {idx} ---")
    print(f"💭 THOUGHT     : {s.thought}")
    print(f"🛠️ ACTION      : {s.tool_name}({s.tool_args})")
    status = s.observation.get('status') if s.observation else 'N/A'
    print(f"👁️ OBSERVATION : Status = {status}")
print("=======================================================")

=================== AUDIT TRACE LOG ===================

--- TURN 1 ---
💭 THOUGHT     : L'ordine supera i 2000€. Devo verificare lo storico acquisti dell'utente e la presenza di chargeback passati.
🛠️ ACTION      : check_customer_history({'user_id': 'USR-4412'})
👁️ OBSERVATION : Status = success

--- TURN 2 ---
💭 THOUGHT     : L'account è stato creato da soli 12 minuti e ha 2 chargeback. Devo verificare la provenienza dell'IP e la presenza di VPN/Tor.
🛠️ ACTION      : verify_geo_ip({'ip_address': '185.220.101.5'})
👁️ OBSERVATION : Status = success

--- TURN 3 ---
💭 THOUGHT     : L'IP proviene da un nodo TOR/VPN a Panama con Rischio 92/100, mentre l'ordine è verso l'Italia e l'account ha precedenti di frode. Procedo con il congelamento preventivo dell'ordine.
🛠️ ACTION      : flag_and_freeze_order({'order_id': 'ORD-9901', 'reason': 'Sospetto utilizzo di credenziali rubate (Account nuovo, IP Tor/VPN, Discrepanza geografica).'})
👁️ OBSERVATION : Status = success

--- TURN 4 ---
💭 THOUGHT 

## Sezione 7: Integrazione con OpenAI API & Function Calling

Ora colleghiamo la logica dell'agente alle API ufficiali di OpenAI, trasformando il planner simulato in un modello linguistico capace di selezionare i tool autonomamente tramite **Function Calling**.

### Schemi JSON dei Tool per OpenAI API

In [11]:
# Definizione dei Tool nel formato compatibile con l'API di OpenAI
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "check_customer_history",
            "description": "Recupera lo storico acquisti, la data di creazione dell'account e i chargeback dell'utente.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {
                        "type": "string",
                        "description": "ID dell'utente e-commerce (es. USR-4412)"
                    }
                },
                "required": ["user_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "verify_geo_ip",
            "description": "Controlla l'indirizzo IP dell'ordine per rilevare l'uso di VPN, Tor e valutare il livello di rischio.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ip_address": {
                        "type": "string",
                        "description": "L'indirizzo IP v4 relativo alla transazione."
                    }
                },
                "required": ["ip_address"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "flag_and_freeze_order",
            "description": "Congela un ordine sospetto per bloccare l'evasione del pacco ed evitare perdite finanziarie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "ID dell'ordine da congelare."
                    },
                    "reason": {
                        "type": "string",
                        "description": "Spiegazione motivata del blocco per l'audit di sicurezza."
                    }
                },
                "required": ["order_id", "reason"]
            }
        }
    }
]

print("✅ Schemi JSON Function Calling pronti per OpenAI.")

✅ Schemi JSON Function Calling pronti per OpenAI.


### Funzione di Esecuzione dell'Agente con OpenAI LLM

In [12]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Assicurati che la chiave API sia presente nell'ambiente
# os.environ["OPENAI_API_KEY"] = "TUA_CHIAVE_API"

def run_openai_fraud_agent(order_id: str):
    """
    Esegue il ciclo di Function Calling automatizzato sfruttando un modello OpenAI.
    """
    if "OPENAI_API_KEY" not in os.environ:
        print("⚠️ OPENAI_API_KEY non trovata nell'ambiente. Inserisci la tua API key per eseguire la chiamata reale.")
        return

    client = OpenAI()
    
    order_info = ORDERS_DB.get(order_id)
    if not order_info:
        print(f"Ordine {order_id} non trovato.")
        return

    system_prompt = (
        "Sei un Fraud Prevention AI Agent per una piattaforma e-commerce. "
        "Il tuo compito è investigare gli ordini contrassegnati come sospetti. "
        "Usa gli strumenti a tua disposizione per raccogliere prove (storico utente, analisi IP) "
        "e, se il rischio è elevato, raccomanda o esegui il congelamento dell'ordine. "
        "Sii preciso, sintetico e basati solo su dati osservati."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Investiga l'ordine {order_id}. Dati iniziali: {json.dumps(order_info)}"}
    ]

    print(f"\n🤖 [OPENAI AGENT] Avvio investigazione per l'ordine {order_id}...")

    max_turns = 5
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        # Controlla se il modello intende chiamare una funzione
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                
                print(f"\n🎯 [LLM CALLS TOOL]: {fn_name}")
                print(f"   Arguments: {fn_args}")

                # Esecuzione del tool locale
                if fn_name in TOOL_REGISTRY:
                    tool_func = TOOL_REGISTRY[fn_name]["func"]
                    tool_output = tool_func(**fn_args)
                else:
                    tool_output = {"error": "Tool non trovato"}

                # Invio del risultato dell'osservazione al modello
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": fn_name,
                    "content": json.dumps(tool_output)
                })
        else:
            # Se il modello non chiama più tool, ha terminato la sua analisi
            print("\n🏁 [AGENTE COMPLETATO]:")
            print(response_message.content)
            break

load_dotenv()
run_openai_fraud_agent("ORD-9901")


🤖 [OPENAI AGENT] Avvio investigazione per l'ordine ORD-9901...

🎯 [LLM CALLS TOOL]: check_customer_history
   Arguments: {'user_id': 'USR-4412'}

🎯 [LLM CALLS TOOL]: verify_geo_ip
   Arguments: {'ip_address': '185.220.101.5'}

🎯 [LLM CALLS TOOL]: flag_and_freeze_order
   Arguments: {'order_id': 'ORD-9901', 'reason': 'Sospetto utilizzo di credenziali rubate (Account nuovo, IP Tor/VPN, Discrepanza geografica, precedenti chargeback). Rischio di frode elevato.'}


C:\Users\Allysa\AppData\Local\Temp\ipykernel_26008\1293588318.py:81: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()



🏁 [AGENTE COMPLETATO]:
L'ordine **ORD-9901** è stato con successo congelato. 

### Dettagli della congelazione:
- **Nuovo stato**: FROZEN_FRAUD_RISK
- **Motivazione**: Sospetto utilizzo di credenziali rubate (Account nuovo, IP Tor/VPN, Discrepanza geografica, precedenti chargeback). Rischio di frode elevato.
- **Timestamp**: 2026-10-02T10:17:43.411861

Se necessiti di ulteriori approfondimenti o azioni, sono a tua disposizione.
